# Defensive security dataset ingestion
This notebook ingests local approved JSON/JSONL sources, preserves provenance, validates the canonical schema, and writes `datasets/processed/security_dataset.jsonl`. It does not download or invent records silently.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

RAW_DIR = Path(os.environ.get('AASP_DATASET_RAW_DIR', 'datasets/raw'))
OUTPUT_PATH = Path(os.environ.get('AASP_DATASET_OUTPUT', 'datasets/processed/security_dataset.jsonl'))
MOUNT_DRIVE = os.environ.get('AASP_MOUNT_DRIVE', '0') == '1'
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print('Google Drive mounted.')
    except ImportError:
        print('Drive requested outside Colab; continuing with local paths.')
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f'Raw source directory: {RAW_DIR.resolve()}')
print(f'Output path: {OUTPUT_PATH.resolve()}')

In [ ]:
VALID_SEVERITIES = {'INFO', 'LOW', 'MEDIUM', 'HIGH', 'CRITICAL'}
REQUIRED_EXPECTED = {'classification', 'severity', 'confidence', 'cwe', 'owasp', 'explanation', 'remediation'}

def read_records(path):
    if path.suffix.lower() == '.jsonl':
        records = []
        for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), start=1):
            if line.strip():
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f'{path}:{line_number}: malformed JSON: {exc}') from exc
        return records
    payload = json.loads(path.read_text(encoding='utf-8'))
    return payload if isinstance(payload, list) else [payload]

def validate_record(record, source, index):
    if not isinstance(record, dict):
        raise ValueError(f'{source} record {index} is not an object')
    for field in ('sample_id', 'task_type', 'evidence', 'expected'):
        if field not in record:
            raise ValueError(f'{source} record {index} is missing {field}')
    if not isinstance(record['sample_id'], str) or not record['sample_id'].strip():
        raise ValueError(f'{source} record {index} has an invalid sample_id')
    if not isinstance(record['task_type'], str) or not record['task_type'].strip():
        raise ValueError(f'{source} record {index} has an invalid task_type')
    if not isinstance(record['evidence'], dict) or not record['evidence']:
        raise ValueError(f'{source} record {index} requires non-empty evidence')
    expected = record['expected']
    if not isinstance(expected, dict) or not REQUIRED_EXPECTED.issubset(expected):
        raise ValueError(f'{source} record {index} has an incomplete expected object')
    if expected['severity'] not in VALID_SEVERITIES:
        raise ValueError(f'{source} record {index} has invalid severity {expected["severity"]}')
    confidence = expected['confidence']
    if not isinstance(confidence, (int, float)) or isinstance(confidence, bool) or not 0 <= confidence <= 1:
        raise ValueError(f'{source} record {index} has invalid confidence')
    if not all(isinstance(item, str) and item.startswith('CWE-') for item in expected['cwe']):
        raise ValueError(f'{source} record {index} has invalid CWE values')
    return {
        'sample_id': record['sample_id'].strip(),
        'task_type': record['task_type'].strip(),
        'evidence': record['evidence'],
        'expected': {key: expected[key] for key in REQUIRED_EXPECTED},
        'provenance': {
            'source_file': str(source),
            'ingested_at': datetime.now(timezone.utc).isoformat(),
        },
    }

In [ ]:
source_paths = sorted([*RAW_DIR.glob('*.json'), *RAW_DIR.glob('*.jsonl')])
if not source_paths:
    raise RuntimeError(
        f'No approved .json or .jsonl sources found in {RAW_DIR}. '
        'Add provenance-reviewed defensive records before running ingestion.'
    )
print(f'Found {len(source_paths)} approved source file(s).')
normalized = []
seen_ids = set()
for source_path in source_paths:
    records = read_records(source_path)
    print(f'Read {len(records)} record(s) from {source_path}')
    for index, record in enumerate(records, start=1):
        canonical = validate_record(record, source_path, index)
        if canonical['sample_id'] in seen_ids:
            raise ValueError(f'Duplicate sample_id during ingestion: {canonical["sample_id"]}')
        seen_ids.add(canonical['sample_id'])
        normalized.append(canonical)

temporary_path = OUTPUT_PATH.with_suffix(OUTPUT_PATH.suffix + '.partial')
with temporary_path.open('w', encoding='utf-8') as handle:
    for record in normalized:
        handle.write(json.dumps(record, sort_keys=True) + '\n')
temporary_path.replace(OUTPUT_PATH)

written = [json.loads(line) for line in OUTPUT_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
if len(written) != len(normalized) or not written:
    raise RuntimeError('Output artifact validation failed.')
print(f'Validated {len(written)} canonical record(s).')
print(f'Artifact: {OUTPUT_PATH.resolve()}')